# VisDrone — Multi-Object Tracking with Re-ID
### YOLOv5 · YOLOv8 · YOLOv11 | Train · Validate · Test

---

**Dataset:** [VisDrone2019](https://github.com/VisDrone/VisDrone-Dataset) — 288 video clips, 261,908 frames, 10,209 static images, 2.6M+ bounding boxes  
**Classes:** pedestrian · people · bicycle · car · van · truck · tricycle · awning-tricycle · bus · motor  
**Tasks covered:** Detection → Tracking (ByteTrack / BoT-SORT) → Re-Identification (OSNet / CLIP-ReID)  

```
Pipeline
  VisDrone frames
      │
      ▼
  YOLO Detector  (v5 / v8 / v11)
      │  bboxes + confidences
      ▼
  Multi-Object Tracker  (ByteTrack / BoT-SORT)
      │  track IDs
      ▼
  Re-ID Embedder  (OSNet / torchreid)
      │  appearance features
      ▼
  Stable long-term IDs
```


## 📋 Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Dataset Download & Preparation](#2-dataset-download--preparation)
3. [Configuration & YAML](#3-configuration--yaml)
4. [YOLOv5 — Train · Validate · Test](#4-yolov5--train--validate--test)
5. [YOLOv8 — Train · Validate · Test](#5-yolov8--train--validate--test)
6. [YOLOv11 — Train · Validate · Test](#6-yolov11--train--validate--test)
7. [Multi-Object Tracking Setup (ByteTrack / BoT-SORT)](#7-multi-object-tracking-setup)
8. [Re-Identification (OSNet / torchreid)](#8-re-identification)
9. [Full MOT + ReID Pipeline (per model)](#9-full-mot--reid-pipeline)
10. [Evaluation & Metrics](#10-evaluation--metrics)
11. [Visualization](#11-visualization)
12. [Model Comparison Summary](#12-model-comparison-summary)

---
## 1. Environment Setup

In [ ]:
# ── Check GPU ──────────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No NVIDIA GPU detected — CPU-only mode')

import torch
print(f"\nPyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"Device  : {'GPU ✅  ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ⚠️ '}")

In [ ]:
# ── Install core dependencies ─────────────────────────────────────────────
!pip install -q ultralytics          # YOLOv8 + v11
!pip install -q supervision          # tracking visualisation helpers
!pip install -q lap                  # Linear Assignment Problem (ByteTrack dep)
!pip install -q gdown                # Google Drive downloader
!pip install -q Pillow matplotlib seaborn pandas tqdm

# YOLOv5 (standalone repo)
import os
if not os.path.exists('yolov5'):
    !git clone -q https://github.com/ultralytics/yolov5.git
    !pip install -q -r yolov5/requirements.txt
print('YOLOv5 cloned ✅')

# BoxMOT — unified tracker (ByteTrack, BoT-SORT, StrongSORT, etc.)
if not os.path.exists('boxmot'):
    !git clone -q https://github.com/mikel-brostrom/boxmot.git
    !pip install -q -e boxmot
print('BoxMOT cloned ✅')

# torchreid — Re-Identification
if not os.path.exists('deep-person-reid'):
    !git clone -q https://github.com/KaiyangZhou/deep-person-reid.git
    %cd deep-person-reid
    !pip install -q -r requirements.txt
    !python setup.py develop -q
    %cd ..
print('torchreid installed ✅')

In [ ]:
# ── Standard imports ─────────────────────────────────────────────────────
import os, sys, shutil, time, json, warnings
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import hsv_to_rgb
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT   = Path('.').resolve()
print(f'Working directory : {ROOT}')
print(f'Device            : {DEVICE}')

---
## 2. Dataset Download & Preparation

In [ ]:
# ── Download VisDrone via Ultralytics auto-download ───────────────────────
# This downloads ~2.3 GB and converts annotations to YOLO format automatically.

from ultralytics import YOLO

# Trigger dataset download by running a short dummy train
# Alternatively: ultralytics will auto-download when data=VisDrone.yaml is used.
print('Downloading VisDrone dataset (≈2.3 GB) …')
print('This may take 5-15 minutes depending on your connection.\n')

# Manually trigger download using ultralytics downloader
from ultralytics.utils.downloads import download
from ultralytics.utils import ASSETS_URL, TQDM as ULTQDM

DATASET_DIR = ROOT / 'datasets' / 'VisDrone'
DATASET_DIR.mkdir(parents=True, exist_ok=True)

urls = [
    f"{ASSETS_URL}/VisDrone2019-DET-train.zip",
    f"{ASSETS_URL}/VisDrone2019-DET-val.zip",
    f"{ASSETS_URL}/VisDrone2019-DET-test-dev.zip",
]
download(urls, dir=DATASET_DIR, threads=3)
print('\n✅ Download complete')

In [ ]:
# ── Convert VisDrone annotations → YOLO format ────────────────────────────
from PIL import Image as PILImage

def visdrone2yolo(dataset_dir, split, source_name=None):
    """Convert VisDrone .txt annotations to YOLO normalised xywh format."""
    dataset_dir = Path(dataset_dir)
    source_dir  = dataset_dir / (source_name or f'VisDrone2019-DET-{split}')
    images_dir  = dataset_dir / 'images' / split
    labels_dir  = dataset_dir / 'labels' / split
    labels_dir.mkdir(parents=True, exist_ok=True)

    src_imgs = source_dir / 'images'
    if src_imgs.exists():
        images_dir.mkdir(parents=True, exist_ok=True)
        for img in src_imgs.glob('*.jpg'):
            dst = images_dir / img.name
            if not dst.exists():
                img.rename(dst)

    anno_dir = source_dir / 'annotations'
    for f in tqdm(list(anno_dir.glob('*.txt')), desc=f'Converting {split}'):
        img_path = images_dir / f.with_suffix('.jpg').name
        if not img_path.exists():
            continue
        w_img, h_img = PILImage.open(img_path).size
        dw, dh = 1.0 / w_img, 1.0 / h_img
        lines = []
        with open(f, encoding='utf-8') as fp:
            for row in [x.split(',') for x in fp.read().strip().splitlines()]:
                if row[4] != '0':          # skip ignored regions
                    x, y, w, h = map(int, row[:4])
                    cls = int(row[5]) - 1  # 0-indexed
                    xc = (x + w / 2) * dw
                    yc = (y + h / 2) * dh
                    wn = w * dw
                    hn = h * dh
                    lines.append(f'{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n')
        (labels_dir / f.name).write_text(''.join(lines), encoding='utf-8')

    if source_dir.exists():
        shutil.rmtree(source_dir)
    print(f'  ✅ {split} converted')

splits = {
    'VisDrone2019-DET-train'   : 'train',
    'VisDrone2019-DET-val'     : 'val',
    'VisDrone2019-DET-test-dev': 'test',
}

for folder, split in splits.items():
    src = DATASET_DIR / folder
    if src.exists():
        visdrone2yolo(DATASET_DIR, split, folder)
    else:
        print(f'  ⏭️  {split} already converted or not found — skipping')

# Verify
for split in ['train', 'val', 'test']:
    imgs = list((DATASET_DIR / 'images' / split).glob('*.jpg'))
    lbls = list((DATASET_DIR / 'labels' / split).glob('*.txt'))
    print(f'{split:5s}: {len(imgs):5d} images | {len(lbls):5d} labels')

In [ ]:
# ── Explore dataset statistics ────────────────────────────────────────────
CLASS_NAMES = [
    'pedestrian','people','bicycle','car','van',
    'truck','tricycle','awning-tricycle','bus','motor'
]

def count_labels(label_dir):
    counts = np.zeros(10, dtype=int)
    for f in Path(label_dir).glob('*.txt'):
        for line in f.read_text().strip().splitlines():
            cls = int(line.split()[0])
            counts[cls] += 1
    return counts

train_counts = count_labels(DATASET_DIR / 'labels/train')
val_counts   = count_labels(DATASET_DIR / 'labels/val')

df_counts = pd.DataFrame({'Class': CLASS_NAMES, 'Train': train_counts, 'Val': val_counts})

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, split, col in zip(axes, ['Train', 'Val'], ['steelblue', 'coral']):
    bars = ax.bar(df_counts['Class'], df_counts[split], color=col, edgecolor='white')
    ax.set_title(f'{split} Class Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Class'); ax.set_ylabel('Instance Count')
    ax.tick_params(axis='x', rotation=40)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+200, f'{h:,}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(df_counts.to_string(index=False))

---
## 3. Configuration & YAML

In [ ]:
# ── Write VisDrone.yaml ───────────────────────────────────────────────────
import yaml

VISDRONE_YAML = ROOT / 'VisDrone.yaml'

cfg = {
    'path'  : str(DATASET_DIR),
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : 10,
    'names' : CLASS_NAMES,
}

with open(VISDRONE_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'Wrote {VISDRONE_YAML}')
print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
# ── Global training hyperparameters ──────────────────────────────────────
EPOCHS    = 100     # reduce to 10-20 for quick smoke-test
IMGSZ     = 640     # 1280 for best accuracy on small objects (VisDrone)
BATCH     = 16      # reduce if OOM (e.g. 8, 4)
WORKERS   = 4
PATIENCE  = 20      # early stopping patience
CONF_THRES = 0.25
IOU_THRES  = 0.45

print(f'Epochs  : {EPOCHS}')
print(f'ImgSz   : {IMGSZ}')
print(f'Batch   : {BATCH}')
print(f'Device  : {DEVICE}')

---
## 4. YOLOv5 — Train · Validate · Test

In [ ]:
# ── 4.1  YOLOv5 — Training ────────────────────────────────────────────────
# YOLOv5 is trained via its CLI (train.py)

V5_WEIGHTS = 'yolov5s.pt'   # options: yolov5n, yolov5s, yolov5m, yolov5l, yolov5x
V5_PROJECT = 'runs/v5'
V5_NAME    = 'visdrone_train'

print('── YOLOv5 Training ────────────────────────────────────────')
!python yolov5/train.py \
    --img       {IMGSZ} \
    --batch     {BATCH} \
    --epochs    {EPOCHS} \
    --data      {VISDRONE_YAML} \
    --weights   {V5_WEIGHTS} \
    --project   {V5_PROJECT} \
    --name      {V5_NAME} \
    --patience  {PATIENCE} \
    --workers   {WORKERS} \
    --device    {0 if DEVICE=='cuda' else 'cpu'} \
    --exist-ok

In [ ]:
# ── 4.2  YOLOv5 — Validation ─────────────────────────────────────────────
V5_BEST = f'{V5_PROJECT}/{V5_NAME}/weights/best.pt'

print('── YOLOv5 Validation ──────────────────────────────────────')
!python yolov5/val.py \
    --weights   {V5_BEST} \
    --data      {VISDRONE_YAML} \
    --img       {IMGSZ} \
    --batch     {BATCH} \
    --conf      {CONF_THRES} \
    --iou       {IOU_THRES} \
    --task      val \
    --project   {V5_PROJECT} \
    --name      visdrone_val \
    --exist-ok \
    --verbose

In [ ]:
# ── 4.3  YOLOv5 — Testing ────────────────────────────────────────────────
print('── YOLOv5 Test ────────────────────────────────────────────')
!python yolov5/val.py \
    --weights   {V5_BEST} \
    --data      {VISDRONE_YAML} \
    --img       {IMGSZ} \
    --batch     {BATCH} \
    --conf      {CONF_THRES} \
    --iou       {IOU_THRES} \
    --task      test \
    --project   {V5_PROJECT} \
    --name      visdrone_test \
    --exist-ok \
    --save-txt \
    --save-conf \
    --verbose

In [ ]:
# ── 4.4  YOLOv5 — Parse results ──────────────────────────────────────────
def parse_v5_results(run_dir):
    csv_path = Path(run_dir) / 'results.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        return df
    return None

v5_results = parse_v5_results(f'{V5_PROJECT}/{V5_NAME}')

if v5_results is not None:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    metrics = [
        ('train/box_loss', 'Train Box Loss', 'tomato'),
        ('train/obj_loss', 'Train Obj Loss', 'orange'),
        ('train/cls_loss', 'Train Cls Loss', 'gold'),
        ('metrics/precision', 'Precision', 'steelblue'),
        ('metrics/recall',    'Recall',    'forestgreen'),
        ('metrics/mAP_0.5',   'mAP@0.5',  'purple'),
    ]
    for ax, (col, title, color) in zip(axes.flat, metrics):
        if col in v5_results.columns:
            ax.plot(v5_results[col], color=color, linewidth=2)
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
    fig.suptitle('YOLOv5 Training Curves — VisDrone', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('v5_training_curves.png', dpi=150)
    plt.show()
else:
    print('results.csv not found — training may not have completed yet')

---
## 5. YOLOv8 — Train · Validate · Test

In [ ]:
# ── 5.1  YOLOv8 — Training ────────────────────────────────────────────────
from ultralytics import YOLO

V8_MODEL  = 'yolov8s.pt'   # options: yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
V8_PROJECT = 'runs/v8'
V8_NAME    = 'visdrone_train'

print('── YOLOv8 Training ────────────────────────────────────────')
model_v8 = YOLO(V8_MODEL)

v8_train_results = model_v8.train(
    data        = str(VISDRONE_YAML),
    epochs      = EPOCHS,
    imgsz       = IMGSZ,
    batch       = BATCH,
    workers     = WORKERS,
    patience    = PATIENCE,
    device      = DEVICE,
    project     = V8_PROJECT,
    name        = V8_NAME,
    exist_ok    = True,
    # VisDrone-specific augmentations for small objects
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.1,
    scale       = 0.5,
    translate   = 0.1,
    degrees     = 0.0,
    shear       = 0.0,
)

print(f'\nBest mAP50   : {v8_train_results.results_dict.get("metrics/mAP50(B)", "N/A"):.4f}')
print(f'Best mAP50-95: {v8_train_results.results_dict.get("metrics/mAP50-95(B)", "N/A"):.4f}')

In [ ]:
# ── 5.2  YOLOv8 — Validation ─────────────────────────────────────────────
V8_BEST = f'{V8_PROJECT}/{V8_NAME}/weights/best.pt'

model_v8_best = YOLO(V8_BEST)

print('── YOLOv8 Validation ──────────────────────────────────────')
v8_val_metrics = model_v8_best.val(
    data    = str(VISDRONE_YAML),
    split   = 'val',
    imgsz   = IMGSZ,
    batch   = BATCH,
    conf    = CONF_THRES,
    iou     = IOU_THRES,
    device  = DEVICE,
    project = V8_PROJECT,
    name    = 'visdrone_val',
    exist_ok= True,
    verbose = True,
)

print(f'\nValidation mAP@0.5    : {v8_val_metrics.box.map50:.4f}')
print(f'Validation mAP@0.5-95 : {v8_val_metrics.box.map:.4f}')
print(f'Precision             : {v8_val_metrics.box.mp:.4f}')
print(f'Recall                : {v8_val_metrics.box.mr:.4f}')

In [ ]:
# ── 5.3  YOLOv8 — Testing ────────────────────────────────────────────────
print('── YOLOv8 Test ────────────────────────────────────────────')
v8_test_metrics = model_v8_best.val(
    data    = str(VISDRONE_YAML),
    split   = 'test',
    imgsz   = IMGSZ,
    batch   = BATCH,
    conf    = CONF_THRES,
    iou     = IOU_THRES,
    device  = DEVICE,
    save_txt = True,
    save_conf= True,
    project = V8_PROJECT,
    name    = 'visdrone_test',
    exist_ok= True,
)

print(f'\nTest mAP@0.5    : {v8_test_metrics.box.map50:.4f}')
print(f'Test mAP@0.5-95 : {v8_test_metrics.box.map:.4f}')

In [ ]:
# ── 5.4  YOLOv8 — Per-class metrics table ────────────────────────────────
if hasattr(v8_val_metrics, 'box'):
    ap_per_class = v8_val_metrics.box.ap_class_index
    ap50_per_class = v8_val_metrics.box.ap50
    p_per_class    = v8_val_metrics.box.p
    r_per_class    = v8_val_metrics.box.r

    df_cls = pd.DataFrame({
        'Class'    : [CLASS_NAMES[i] for i in ap_per_class],
        'Precision': p_per_class,
        'Recall'   : r_per_class,
        'AP@0.5'   : ap50_per_class,
    }).sort_values('AP@0.5', ascending=False)

    print('\nYOLOv8 Per-Class Validation Metrics')
    print(df_cls.to_string(index=False, float_format='%.4f'))

    # Heatmap
    fig, ax = plt.subplots(figsize=(10, 5))
    df_heat = df_cls.set_index('Class')[['Precision','Recall','AP@0.5']]
    sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax,
                linewidths=0.5, cbar_kws={'label': 'Score'})
    ax.set_title('YOLOv8 Per-Class Metrics — VisDrone', fontweight='bold')
    plt.tight_layout()
    plt.savefig('v8_per_class_metrics.png', dpi=150)
    plt.show()

---
## 6. YOLOv11 — Train · Validate · Test

In [ ]:
# ── 6.1  YOLOv11 — Training ───────────────────────────────────────────────
# YOLOv11 is available in Ultralytics >= 8.3.0

V11_MODEL  = 'yolo11s.pt'  # options: yolo11n, yolo11s, yolo11m, yolo11l, yolo11x
V11_PROJECT = 'runs/v11'
V11_NAME    = 'visdrone_train'

print('── YOLOv11 Training ───────────────────────────────────────')
model_v11 = YOLO(V11_MODEL)

v11_train_results = model_v11.train(
    data        = str(VISDRONE_YAML),
    epochs      = EPOCHS,
    imgsz       = IMGSZ,
    batch       = BATCH,
    workers     = WORKERS,
    patience    = PATIENCE,
    device      = DEVICE,
    project     = V11_PROJECT,
    name        = V11_NAME,
    exist_ok    = True,
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.1,
)

print(f'\nBest mAP50   : {v11_train_results.results_dict.get("metrics/mAP50(B)", "N/A"):.4f}')
print(f'Best mAP50-95: {v11_train_results.results_dict.get("metrics/mAP50-95(B)", "N/A"):.4f}')

In [ ]:
# ── 6.2  YOLOv11 — Validation ────────────────────────────────────────────
V11_BEST = f'{V11_PROJECT}/{V11_NAME}/weights/best.pt'
model_v11_best = YOLO(V11_BEST)

print('── YOLOv11 Validation ─────────────────────────────────────')
v11_val_metrics = model_v11_best.val(
    data     = str(VISDRONE_YAML),
    split    = 'val',
    imgsz    = IMGSZ,
    batch    = BATCH,
    conf     = CONF_THRES,
    iou      = IOU_THRES,
    device   = DEVICE,
    project  = V11_PROJECT,
    name     = 'visdrone_val',
    exist_ok = True,
    verbose  = True,
)

print(f'\nValidation mAP@0.5    : {v11_val_metrics.box.map50:.4f}')
print(f'Validation mAP@0.5-95 : {v11_val_metrics.box.map:.4f}')

In [ ]:
# ── 6.3  YOLOv11 — Testing ───────────────────────────────────────────────
print('── YOLOv11 Test ───────────────────────────────────────────')
v11_test_metrics = model_v11_best.val(
    data     = str(VISDRONE_YAML),
    split    = 'test',
    imgsz    = IMGSZ,
    batch    = BATCH,
    conf     = CONF_THRES,
    iou      = IOU_THRES,
    device   = DEVICE,
    save_txt = True,
    save_conf= True,
    project  = V11_PROJECT,
    name     = 'visdrone_test',
    exist_ok = True,
)

print(f'\nTest mAP@0.5    : {v11_test_metrics.box.map50:.4f}')
print(f'Test mAP@0.5-95 : {v11_test_metrics.box.map:.4f}')

---
## 7. Multi-Object Tracking Setup

We use **BoxMOT** — a unified Python library that wraps ByteTrack, BoT-SORT, StrongSORT, and others.  
The tracker ingests YOLO detections and returns persistent track IDs.

```
Frame  →  YOLO Detector  →  [x1,y1,x2,y2,conf,cls]  →  Tracker  →  [x1,y1,x2,y2,track_id,conf,cls]
```

In [ ]:
# ── Tracker configuration ─────────────────────────────────────────────────
sys.path.insert(0, str(ROOT / 'boxmot'))

from boxmot import ByteTrack, BotSort

def build_tracker(tracker_type='bytetrack', reid_weights=None):
    """
    tracker_type : 'bytetrack' | 'botsort'
    reid_weights : path to OSNet weights (used by BoT-SORT for appearance)
    """
    if tracker_type == 'bytetrack':
        tracker = ByteTrack(
            track_thresh     = 0.25,
            track_buffer     = 30,
            match_thresh     = 0.8,
            frame_rate       = 25,
        )
    elif tracker_type == 'botsort':
        tracker = BotSort(
            reid_weights     = Path(reid_weights) if reid_weights else None,
            device           = DEVICE,
            half             = False,
            track_high_thresh= 0.5,
            track_low_thresh = 0.1,
            new_track_thresh = 0.6,
            track_buffer     = 30,
            match_thresh     = 0.8,
            proximity_thresh = 0.5,
            appearance_thresh= 0.25,
            with_reid        = reid_weights is not None,
        )
    return tracker

print('Tracker factory ready')

---
## 8. Re-Identification (OSNet / torchreid)

Re-ID adds appearance embeddings so the tracker can **re-associate lost tracks** after occlusion.  
We use **OSNet** (Omni-Scale Network) — compact and accurate for person/vehicle ReID.

In [ ]:
# ── Download pretrained OSNet weights ─────────────────────────────────────
import torchreid

REID_WEIGHTS_DIR = ROOT / 'reid_weights'
REID_WEIGHTS_DIR.mkdir(exist_ok=True)

# Build OSNet model using torchreid
reid_model = torchreid.models.build_model(
    name       = 'osnet_x1_0',
    num_classes= 1000,         # pretrained on Market-1501 (1000 IDs)
    pretrained = True,
).to(DEVICE).eval()

print(f'OSNet parameters : {sum(p.numel() for p in reid_model.parameters())/1e6:.1f}M')

# ReID preprocessing
import torchvision.transforms as T

REID_TRANSFORM = T.Compose([
    T.Resize((256, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

def extract_reid_features(image_bgr, boxes):
    """
    Extract OSNet appearance embeddings for each bounding box crop.
    
    Args:
        image_bgr : full frame (H,W,3) BGR numpy array
        boxes     : Nx4 array of [x1,y1,x2,y2] pixel coordinates
    Returns:
        features  : Nx512 float32 numpy array (L2-normalised)
    """
    if len(boxes) == 0:
        return np.zeros((0, 512), dtype=np.float32)
    
    crops = []
    h, w = image_bgr.shape[:2]
    for x1, y1, x2, y2 in boxes.astype(int):
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            crops.append(torch.zeros(3, 256, 128))
            continue
        crop = image_bgr[y1:y2, x1:x2, ::-1]  # BGR → RGB
        crop = PILImage.fromarray(crop.astype(np.uint8))
        crops.append(REID_TRANSFORM(crop))

    batch = torch.stack(crops).to(DEVICE)
    with torch.no_grad():
        features = reid_model(batch)           # (N, 512)
        features = F.normalize(features, dim=1)
    return features.cpu().numpy()

print('ReID feature extractor ready')

In [ ]:
# ── Cosine-similarity ReID matcher ────────────────────────────────────────
class ReIDMemory:
    """
    Simple gallery memory that stores appearance features per track ID.
    Used to re-identify tracks that were lost and reappear.
    """
    def __init__(self, max_gallery=200, similarity_threshold=0.6, momentum=0.9):
        self.gallery     = {}   # track_id → feature vector
        self.max_gallery = max_gallery
        self.sim_thresh  = similarity_threshold
        self.momentum    = momentum

    def update(self, track_id, feature):
        if track_id in self.gallery:
            # Exponential moving average
            self.gallery[track_id] = (
                self.momentum * self.gallery[track_id] +
                (1 - self.momentum) * feature
            )
        else:
            if len(self.gallery) >= self.max_gallery:
                oldest = next(iter(self.gallery))
                del self.gallery[oldest]
            self.gallery[track_id] = feature.copy()

    def find_best_match(self, feature):
        """
        Return (best_track_id, similarity) or (None, 0) if no match above threshold.
        """
        if not self.gallery:
            return None, 0.0
        ids      = list(self.gallery.keys())
        gallery  = np.stack(list(self.gallery.values()))  # (N, D)
        sims     = gallery @ feature                      # cosine sim (L2 normalised)
        best_idx = int(np.argmax(sims))
        best_sim = float(sims[best_idx])
        if best_sim >= self.sim_thresh:
            return ids[best_idx], best_sim
        return None, best_sim

print('ReIDMemory class defined')

---
## 9. Full MOT + ReID Pipeline (per model)

In [ ]:
# ── Core tracking + ReID loop ──────────────────────────────────────────────
def run_mot_reid(
    detector_weights : str,
    video_path       : str,
    output_path      : str,
    tracker_type     : str = 'bytetrack',
    use_reid         : bool = True,
    max_frames       : int = None,
):
    """
    Full pipeline: YOLO detection → MOT (ByteTrack/BoT-SORT) → ReID embedding.

    Args:
        detector_weights : path to YOLO .pt weights
        video_path       : input video file
        output_path      : output annotated video file
        tracker_type     : 'bytetrack' | 'botsort'
        use_reid         : attach OSNet appearance features
        max_frames       : process only first N frames (None = all)

    Returns:
        mot_results : list of dicts per frame
    """
    detector  = YOLO(detector_weights)
    tracker   = build_tracker(tracker_type)
    reid_mem  = ReIDMemory() if use_reid else None

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps, (W, H)
    )

    # Colour palette for track IDs
    rng = np.random.default_rng(42)
    palette = {}

    def get_colour(tid):
        if tid not in palette:
            h = rng.uniform(0, 1)
            palette[tid] = tuple((np.array(hsv_to_rgb([h, 0.9, 0.95])) * 255).astype(int).tolist())
        return palette[tid]

    mot_results = []

    for frame_idx in tqdm(range(total), desc=f'Tracking [{Path(video_path).stem}]'):
        ret, frame = cap.read()
        if not ret:
            break

        # ── Detection ───────────────────────────────────────────────────
        dets = detector(
            frame,
            imgsz   = IMGSZ,
            conf    = CONF_THRES,
            iou     = IOU_THRES,
            verbose = False,
        )[0]

        boxes_xyxy = dets.boxes.xyxy.cpu().numpy()   # (N,4)
        confs      = dets.boxes.conf.cpu().numpy()   # (N,)
        classes    = dets.boxes.cls.cpu().numpy()    # (N,)

        det_arr = np.column_stack(
            [boxes_xyxy, confs, classes]
        ) if len(boxes_xyxy) else np.empty((0, 6))

        # ── Tracking ────────────────────────────────────────────────────
        tracks = tracker.update(det_arr, frame)      # (M, 8): x1,y1,x2,y2,id,conf,cls,idx

        frame_data = {'frame': frame_idx, 'tracks': []}

        if len(tracks) == 0:
            writer.write(frame)
            mot_results.append(frame_data)
            continue

        track_boxes = tracks[:, :4].astype(int)
        track_ids   = tracks[:, 4].astype(int)
        track_confs = tracks[:, 5]
        track_clses = tracks[:, 6].astype(int)

        # ── ReID ────────────────────────────────────────────────────────
        reid_features = None
        if use_reid and reid_mem is not None:
            reid_features = extract_reid_features(frame, track_boxes)
            for i, (tid, feat) in enumerate(zip(track_ids, reid_features)):
                reid_mem.update(tid, feat)

        # ── Annotate ────────────────────────────────────────────────────
        for i, (box, tid, conf, cls) in enumerate(zip(track_boxes, track_ids, track_confs, track_clses)):
            x1, y1, x2, y2 = box
            colour = get_colour(tid)
            label  = f'{CLASS_NAMES[cls]} #{tid} {conf:.2f}'
            cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
            cv2.rectangle(frame, (x1, y1-th-6), (x1+tw+4, y1), colour, -1)
            cv2.putText(frame, label, (x1+2, y1-3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)

            track_entry = {
                'track_id': int(tid),
                'class'   : CLASS_NAMES[cls],
                'box'     : [int(x1), int(y1), int(x2), int(y2)],
                'conf'    : float(conf),
            }
            if reid_features is not None:
                track_entry['reid_feat'] = reid_features[i].tolist()
            frame_data['tracks'].append(track_entry)

        # Frame info overlay
        cv2.putText(frame, f'Frame {frame_idx}  Tracks: {len(tracks)}',
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        writer.write(frame)
        mot_results.append(frame_data)

    cap.release()
    writer.release()
    print(f'\n✅  Saved to {output_path}')
    return mot_results

print('run_mot_reid() defined')

In [ ]:
# ── Run pipeline on a VisDrone video for each model ───────────────────────
#
# VisDrone Task-4 (MOT) videos are in VisDrone2019-MOT-train/sequences/
# If you downloaded the MOT split, update VIDEO_DIR below.
# Otherwise we convert a sample image sequence to a video for demonstration.

# ── Sample: convert 50 VisDrone test images to a pseudo-video ─────────────
SAMPLE_IMGS = sorted((DATASET_DIR / 'images/test').glob('*.jpg'))[:50]
DEMO_VIDEO  = 'demo_sequence.mp4'

if SAMPLE_IMGS:
    sample_img = cv2.imread(str(SAMPLE_IMGS[0]))
    H_s, W_s   = sample_img.shape[:2]
    vw = cv2.VideoWriter(DEMO_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), 5, (W_s, H_s))
    for p in SAMPLE_IMGS:
        vw.write(cv2.imread(str(p)))
    vw.release()
    print(f'Created demo video: {DEMO_VIDEO}  ({len(SAMPLE_IMGS)} frames)')
else:
    print('No test images found — point DEMO_VIDEO to your own VisDrone video file')
    DEMO_VIDEO = 'your_visdrone_video.mp4'

In [ ]:
# ── 9.1  YOLOv5 + ByteTrack + ReID ───────────────────────────────────────
print('=== YOLOv5 MOT + ReID ===')
v5_mot_results = run_mot_reid(
    detector_weights = V5_BEST,
    video_path       = DEMO_VIDEO,
    output_path      = 'outputs/v5_mot_reid.mp4',
    tracker_type     = 'bytetrack',
    use_reid         = True,
)

In [ ]:
# ── 9.2  YOLOv8 + ByteTrack + ReID ───────────────────────────────────────
print('=== YOLOv8 MOT + ReID ===')
v8_mot_results = run_mot_reid(
    detector_weights = V8_BEST,
    video_path       = DEMO_VIDEO,
    output_path      = 'outputs/v8_mot_reid.mp4',
    tracker_type     = 'bytetrack',
    use_reid         = True,
)

In [ ]:
# ── 9.3  YOLOv11 + BoT-SORT + ReID ───────────────────────────────────────
print('=== YOLOv11 MOT + ReID ===')
v11_mot_results = run_mot_reid(
    detector_weights = V11_BEST,
    video_path       = DEMO_VIDEO,
    output_path      = 'outputs/v11_mot_reid.mp4',
    tracker_type     = 'botsort',
    use_reid         = True,
)

---
## 10. Evaluation & Metrics

Standard MOT metrics: **MOTA, IDF1, MOTP, MT, ML, FP, FN, ID Sw.**  
Detection metrics: **mAP@0.5, mAP@0.5-95, Precision, Recall**

In [ ]:
# ── MOT evaluation with py-motmetrics ─────────────────────────────────────
!pip install -q motmetrics
import motmetrics as mm

def build_mot_accumulator(mot_results, gt_results):
    """
    Build a motmetrics accumulator from predicted and GT track results.

    mot_results / gt_results : list of dicts
        Each dict: {'frame': int, 'tracks': [{'track_id':int, 'box':[x1,y1,x2,y2]}, ...]}
    """
    acc = mm.MOTAccumulator(auto_id=True)

    pred_dict = {r['frame']: r['tracks'] for r in mot_results}
    gt_dict   = {r['frame']: r['tracks'] for r in gt_results}

    all_frames = sorted(set(pred_dict) | set(gt_dict))

    for fid in all_frames:
        gt_tracks   = gt_dict.get(fid, [])
        pred_tracks = pred_dict.get(fid, [])

        gt_ids   = [t['track_id'] for t in gt_tracks]
        pred_ids = [t['track_id'] for t in pred_tracks]

        def box_iou(b1, b2):
            xa = max(b1[0], b2[0]); ya = max(b1[1], b2[1])
            xb = min(b1[2], b2[2]); yb = min(b1[3], b2[3])
            inter = max(0, xb-xa) * max(0, yb-ya)
            a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
            a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
            union = a1 + a2 - inter + 1e-6
            return inter / union

        if gt_ids and pred_ids:
            dist_mat = np.array([
                [1 - box_iou(g['box'], p['box'])
                 for p in pred_tracks]
                for g in gt_tracks
            ])
        else:
            dist_mat = np.empty((len(gt_ids), len(pred_ids)))

        acc.update(gt_ids, pred_ids, dist_mat)

    return acc


def compute_mot_metrics(acc):
    mh = mm.metrics.create()
    summary = mh.compute(
        acc,
        metrics=['num_frames','mota','motp','idf1',
                 'num_switches','num_fragmentations',
                 'precision','recall','mostly_tracked','mostly_lost'],
        name='Results',
    )
    return summary


# NOTE: Replace `gt_results` with real ground-truth data from VisDrone MOT annotations.
# For demonstration we show the metric computation structure.
print('MOT evaluation functions defined.')
print("To evaluate, call:")
print("  acc = build_mot_accumulator(v8_mot_results, gt_results)")
print("  df  = compute_mot_metrics(acc)")
print("  print(mm.io.render_summary(df, namemap=mm.io.motchallenge_metric_names))")

In [ ]:
# ── Summarise detection metrics across all three models ───────────────────
def safe_get(metrics_obj, attr, default=None):
    try:
        return float(getattr(metrics_obj.box, attr))
    except:
        return default

summary_data = {
    'Model'       : ['YOLOv5s', 'YOLOv8s', 'YOLOv11s'],
    'mAP@0.5'     : [
        None,  # fill from v5 val output
        safe_get(v8_val_metrics, 'map50'),
        safe_get(v11_val_metrics,'map50'),
    ],
    'mAP@0.5-0.95': [
        None,
        safe_get(v8_val_metrics, 'map'),
        safe_get(v11_val_metrics,'map'),
    ],
    'Precision'   : [
        None,
        safe_get(v8_val_metrics, 'mp'),
        safe_get(v11_val_metrics,'mp'),
    ],
    'Recall'      : [
        None,
        safe_get(v8_val_metrics, 'mr'),
        safe_get(v11_val_metrics,'mr'),
    ],
}

df_summary = pd.DataFrame(summary_data)
print('\nDetection Metrics Summary')
print(df_summary.to_string(index=False, float_format='%.4f'))

---
## 11. Visualization

In [ ]:
# ── 11.1  Show sample detections from val set ─────────────────────────────
def visualise_predictions(model_path, img_paths, n_images=6, title=''):
    model = YOLO(model_path)
    rows  = (n_images + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=(18, 5 * rows))
    axes = axes.flat

    for ax, img_path in zip(axes, img_paths[:n_images]):
        results = model(str(img_path), imgsz=IMGSZ, conf=CONF_THRES, verbose=False)[0]
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        for box in results.boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])
            conf= float(box.conf[0])
            rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                      linewidth=1.5, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-3, f'{CLASS_NAMES[cls]} {conf:.2f}',
                    color='lime', fontsize=6, fontweight='bold')

        ax.imshow(img_rgb)
        ax.set_title(Path(img_path).name, fontsize=8)
        ax.axis('off')

    for ax in list(axes)[n_images:]:
        ax.axis('off')

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}_predictions.png', dpi=150)
    plt.show()

val_imgs = sorted((DATASET_DIR / 'images/val').glob('*.jpg'))[:6]

# Uncomment the model you want to visualise:
# visualise_predictions(V5_BEST,  val_imgs, title='YOLOv5 Predictions')
visualise_predictions(V8_BEST,  val_imgs, title='YOLOv8 Predictions')
# visualise_predictions(V11_BEST, val_imgs, title='YOLOv11 Predictions')

In [ ]:
# ── 11.2  Tracking trajectory plot ───────────────────────────────────────
def plot_trajectories(mot_results, title='Track Trajectories', top_k=20):
    """Plot centre-point trajectories for the most frequent track IDs."""
    traj = {}  # track_id → list of (cx, cy)
    for frame_data in mot_results:
        for t in frame_data['tracks']:
            tid = t['track_id']
            b   = t['box']
            cx  = (b[0] + b[2]) / 2
            cy  = (b[1] + b[3]) / 2
            traj.setdefault(tid, []).append((cx, cy))

    # Keep the top-k longest tracks
    top_tids = sorted(traj, key=lambda x: -len(traj[x]))[:top_k]

    fig, ax = plt.subplots(figsize=(12, 8))
    cmap = plt.cm.get_cmap('tab20', len(top_tids))
    for i, tid in enumerate(top_tids):
        pts = np.array(traj[tid])
        ax.plot(pts[:, 0], pts[:, 1], '-o', markersize=2,
                linewidth=1.5, color=cmap(i), label=f'ID {tid}')
        ax.text(pts[-1, 0], pts[-1, 1], str(tid), fontsize=7, color=cmap(i))

    ax.invert_yaxis()
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')
    ax.legend(fontsize=7, ncol=4, loc='upper right')
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}.png', dpi=150)
    plt.show()
    print(f'Total unique tracks: {len(traj)}')

plot_trajectories(v8_mot_results, title='YOLOv8 Track Trajectories')

In [ ]:
# ── 11.3  ReID similarity heatmap (between gallery tracks) ───────────────
def plot_reid_similarity(mot_results, n_tracks=15, title='ReID Cosine Similarity'):
    """Average appearance embedding per track → cosine similarity matrix."""
    track_feats = {}
    for fd in mot_results:
        for t in fd['tracks']:
            if 'reid_feat' in t:
                tid  = t['track_id']
                feat = np.array(t['reid_feat'])
                if tid not in track_feats:
                    track_feats[tid] = []
                track_feats[tid].append(feat)

    if len(track_feats) < 2:
        print('Not enough tracks with ReID features')
        return

    # Average features
    tids  = sorted(track_feats, key=lambda x: -len(track_feats[x]))[:n_tracks]
    feats = np.array([np.mean(track_feats[tid], axis=0) for tid in tids])
    # L2 normalise
    feats = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-6)
    sim   = feats @ feats.T

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(sim, xticklabels=tids, yticklabels=tids,
                annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-1, vmax=1, ax=ax, linewidths=0.3)
    ax.set_title(title, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}.png', dpi=150)
    plt.show()

plot_reid_similarity(v8_mot_results, title='YOLOv8 ReID Similarity Matrix')

In [ ]:
# ── 11.4  Tracks-per-frame plot ───────────────────────────────────────────
def plot_tracks_per_frame(mot_results_dict):
    fig, ax = plt.subplots(figsize=(14, 5))
    colours = {'YOLOv5': 'tomato', 'YOLOv8': 'steelblue', 'YOLOv11': 'seagreen'}
    for name, results in mot_results_dict.items():
        counts = [len(fd['tracks']) for fd in results]
        ax.plot(counts, label=name, color=colours.get(name, 'gray'), linewidth=2)
    ax.set_title('Active Tracks per Frame', fontweight='bold')
    ax.set_xlabel('Frame Index'); ax.set_ylabel('# Active Tracks')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('tracks_per_frame.png', dpi=150)
    plt.show()

plot_tracks_per_frame({
    'YOLOv5' : v5_mot_results,
    'YOLOv8' : v8_mot_results,
    'YOLOv11': v11_mot_results,
})

---
## 12. Model Comparison Summary

In [ ]:
# ── Final comparison table ────────────────────────────────────────────────
def tracking_stats(results):
    total_tracks = set()
    total_dets   = 0
    for fd in results:
        for t in fd['tracks']:
            total_tracks.add(t['track_id'])
            total_dets += 1
    return len(total_tracks), total_dets

v5_ntracks,  v5_ndets  = tracking_stats(v5_mot_results)
v8_ntracks,  v8_ndets  = tracking_stats(v8_mot_results)
v11_ntracks, v11_ndets = tracking_stats(v11_mot_results)

df_comp = pd.DataFrame({
    'Model'             : ['YOLOv5s', 'YOLOv8s', 'YOLOv11s'],
    'Tracker'           : ['ByteTrack', 'ByteTrack', 'BoT-SORT'],
    'ReID'              : ['OSNet', 'OSNet', 'OSNet'],
    'mAP@0.5 (val)'     : [
        'see runs/v5',
        f"{safe_get(v8_val_metrics,'map50'):.4f}"  if safe_get(v8_val_metrics,'map50') else 'N/A',
        f"{safe_get(v11_val_metrics,'map50'):.4f}" if safe_get(v11_val_metrics,'map50') else 'N/A',
    ],
    'mAP@0.5-95 (val)'  : [
        'see runs/v5',
        f"{safe_get(v8_val_metrics,'map'):.4f}"  if safe_get(v8_val_metrics,'map') else 'N/A',
        f"{safe_get(v11_val_metrics,'map'):.4f}" if safe_get(v11_val_metrics,'map') else 'N/A',
    ],
    'Unique Track IDs'  : [v5_ntracks, v8_ntracks, v11_ntracks],
    'Total Detections'  : [v5_ndets,   v8_ndets,   v11_ndets],
})

print('\n' + '='*70)
print('         MODEL COMPARISON — VisDrone MOT + ReID')
print('='*70)
print(df_comp.to_string(index=False))
print('='*70)

# Save to CSV
df_comp.to_csv('model_comparison.csv', index=False)
print('\nSaved model_comparison.csv')

In [ ]:
# ── Bar chart comparison ──────────────────────────────────────────────────
models = ['YOLOv5s', 'YOLOv8s', 'YOLOv11s']
unique_tracks = [v5_ntracks, v8_ntracks, v11_ntracks]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Unique tracks
colours = ['tomato', 'steelblue', 'seagreen']
axes[0].bar(models, unique_tracks, color=colours, edgecolor='white')
axes[0].set_title('Unique Track IDs (lower = less ID fragmentation)',
                  fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(unique_tracks):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Total detections
total_dets = [v5_ndets, v8_ndets, v11_ndets]
axes[1].bar(models, total_dets, color=colours, edgecolor='white')
axes[1].set_title('Total Tracked Detections', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(total_dets):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

fig.suptitle('YOLOv5 vs YOLOv8 vs YOLOv11 — MOT Comparison (VisDrone)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── Save all track results to JSON ────────────────────────────────────────
Path('outputs').mkdir(exist_ok=True)

for name, results in [('v5', v5_mot_results),('v8', v8_mot_results),('v11', v11_mot_results)]:
    out_path = f'outputs/{name}_mot_results.json'
    # Strip large reid_feat arrays for JSON size
    clean = []
    for fd in results:
        clean.append({
            'frame' : fd['frame'],
            'tracks': [{k:v for k,v in t.items() if k != 'reid_feat'}
                       for t in fd['tracks']]
        })
    with open(out_path, 'w') as f:
        json.dump(clean, f)
    print(f'Saved {out_path}')

print('\n🎉 All done! Check the outputs/ directory for annotated videos and JSON results.')

---

## 📌 Quick Reference

| Component | Choice | Notes |
|-----------|--------|-------|
| Detector | YOLOv5s / v8s / v11s | Swap `s→m→l→x` for better accuracy |
| Tracker | ByteTrack | Fast, motion-only. Good for VisDrone |
| Tracker | BoT-SORT | Motion + appearance. Better re-ID |
| ReID | OSNet-x1.0 | 2.2M params, real-time |
| Dataset | VisDrone 2019 | 10 classes, drone imagery |
| Image size | 640 (1280 for best) | Larger → better small objects |

## 📚 Citation

```bibtex
@ARTICLE{9573394,
  author={Zhu, Pengfei and Wen, Longyin and Du, Dawei and Bian, Xiao and Fan, Heng and Hu, Qinghua and Ling, Haibin},
  journal={IEEE Transactions on Pattern Analysis and Machine Intelligence},
  title={Detection and Tracking Meet Drones Challenge},
  year={2021}, doi={10.1109/TPAMI.2021.3119563}
}
```